In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling
from datasets import load_dataset
from accelerate import Accelerator
from tqdm import tqdm
import os

In [ ]:
# 配置参数
config = {
    "model_name": "gpt2",  # 或者使用其他基座模型，如 "meta-llama/Llama-2-7b-hf"
    "dataset_name": "wikitext",
    "dataset_config": "wikitext-2-raw-v1",
    "batch_size": 4,
    "learning_rate": 5e-5,
    "num_epochs": 3,
    "max_length": 512,
    "output_dir": "./trained_model",
}

In [ ]:
def prepare_data(config):
    """
    加载数据集并进行分词处理
    """
    # 1. 加载 tokenizer
    tokenizer = AutoTokenizer.from_pretrained(config["model_name"])
    # 设置 padding token，如果不存在则使用 eos_token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 2. 加载数据集
    dataset = load_dataset(config["dataset_name"], config["dataset_config"])
    
    # 3. 定义分词函数
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=config["max_length"],
            padding="max_length"
        )

    # 4. 对数据集进行分词处理
    tokenized_datasets = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=["text"]  # 移除原始文本列，只保留 input_ids 等
    )

    # 5. 创建数据整理器 (Data Collator)
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False  # False 表示因果语言建模 (CLM)，即 GPT 风格；True 为掩码语言建模 (MLM)，即 BERT 风格
    )

    return tokenized_datasets, data_collator, tokenizer

In [ ]:
def initialize_model(config):
    """
    加载预训练模型
    """
    model = AutoModelForCausalLM.from_pretrained(config["model_name"])
    return model

In [ ]:
def train(config):
    """
    主训练函数
    """
    # 1. 初始化 Accelerator (自动处理设备放置和分布式训练)
    accelerator = Accelerator()

    # 2. 准备数据
    tokenized_datasets, data_collator, tokenizer = prepare_data(config)
    
    # 3. 创建 DataLoader
    train_dataloader = DataLoader(
        tokenized_datasets["train"],
        shuffle=True,
        batch_size=config["batch_size"],
        collate_fn=data_collator
    )

    # 4. 初始化模型
    model = initialize_model(config)

    # 5. 定义优化器
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])

    # 6. 准备所有对象以便 Accelerator 管理
    model, optimizer, train_dataloader = accelerator.prepare(
        model, optimizer, train_dataloader
    )

    # 7. 训练循环
    model.train()
    for epoch in range(config["num_epochs"]):
        print(f"Epoch {epoch + 1}/{config['num_epochs']}")
        progress_bar = tqdm(train_dataloader, disable=not accelerator.is_local_main_process)
        
        for step, batch in enumerate(progress_bar):
            # 前向传播
            outputs = model(**batch)
            loss = outputs.loss
            
            # 反向传播
            accelerator.backward(loss)
            
            # 更新参数
            optimizer.step()
            optimizer.zero_grad()
            
            # 更新进度条
            progress_bar.set_postfix({"loss": loss.item()})
            
            # 可选：定期保存检查点
            if step % 100 == 0 and step != 0:
                accelerator.wait_for_everyone()
                unwrapped_model = accelerator.unwrap_model(model)
                unwrapped_model.save_pretrained(os.path.join(config["output_dir"], f"checkpoint_epoch_{epoch}_step_{step}"))
                tokenizer.save_pretrained(os.path.join(config["output_dir"], f"checkpoint_epoch_{epoch}_step_{step}"))

    # 8. 保存最终模型
    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.save_pretrained(config["output_dir"])
    tokenizer.save_pretrained(config["output_dir"])
    print(f"Model saved to {config['output_dir']}")

if __name__ == "__main__":
    train(config)